# Knowledge Conflict Detection

Detect interference between competing knowledge representations: logit direction conflicts, residual tug-of-war, attention competition, and interference localization.

## Why This Matters

Knowledge conflicts investigates where and how models store factual information. Locating knowledge in specific components (neurons, layers, directions) is essential for understanding hallucination, enabling fact editing, and assessing what a model truly knows.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — Causal tracing to locate factual knowledge

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.knowledge_conflicts import (
    logit_direction_conflicts, residual_tug_of_war,
    attention_competition, interference_localization,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
result = logit_direction_conflicts(model, tokens)
print(f'Total conflicts: {result["n_conflicts"]}')
for c in result['top_conflicts'][:3]:
    print(f'  {c["component_a"]} promotes {c["promotes_a"]} vs '
          f'{c["component_b"]} promotes {c["promotes_b"]} '
          f'(strength={c["conflict_strength"]:.3f})')

In [ ]:
target = int(jnp.argmax(model(tokens)[-1]))
result = residual_tug_of_war(model, tokens, target_token=target)
print(f'Target token: {target}')
print(f'Net logit: {result["net_logit"]:.4f}')
print(f'Promoters:')
for p in result['promoters'][:3]:
    print(f'  {p["component"]}: +{p["contribution"]:.4f}')
print(f'Suppressors:')
for s in result['suppressors'][:3]:
    print(f'  {s["component"]}: {s["contribution"]:.4f}')

In [ ]:
result = attention_competition(model, tokens)
most = result['most_competitive']
print(f'Most competitive: L{most["layer"]}H{most["head"]} '
      f'(ratio={most["competition_ratio"]:.3f})')
for h in result['per_head'][:5]:
    print(f'  L{h["layer"]}H{h["head"]}: ratio={h["competition_ratio"]:.3f}, '
          f'entropy={h["attention_entropy"]:.3f}')

In [ ]:
tokens_b = jnp.array([90, 75, 60, 45, 30, 15, 1])
result = interference_localization(model, tokens, tokens_b)
print(f'Max divergence: {result["max_divergence_component"]}')
for c in result['top_divergent'][:3]:
    print(f'  {c["component"]}: divergence={c["divergence"]:.4f}, '
          f'cos_sim={c["cosine_similarity"]:.3f}')